# Predicción de Ventas Diarias de Retail

Proyecto de regresión listo para **Google Colab**.

**Pregunta:** ¿qué factores influyen en las ventas diarias de un retail?

**Target:** `ventas`

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset de ejemplo para predicción de ventas.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar datos

In [ ]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/tips.csv"
df = pd.read_csv(url)
df["ventas"] = df["total_bill"] * 50
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df.describe().T)

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["ventas"], kde=True, bins=30)
plt.title("Distribución de ventas")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="day", y="ventas", ax=axes[0])
axes[0].set_title("Ventas por día")
sns.boxplot(data=df, x="time", y="ventas", ax=axes[1])
axes[1].set_title("Ventas por momento")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.scatterplot(data=df, x="total_bill", y="ventas", hue="size")
plt.title("Total vs Ventas")
plt.show()

### Hallazgos
- Las ventas dependen de variables numéricas base.
- Diferencias por día y momento.
- Ideal para practicar predicción de ventas de retail.

## 3. Preparar datos

In [ ]:
X = df.drop(columns=["ventas"])
y = df["ventas"]

cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos

In [ ]:
modelos = {
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    resultados.append({
        "modelo": nombre,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred, squared=False),
        "R2": r2_score(y_test, y_pred),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("R2", ascending=False)

## 5. Feature importance

In [ ]:
rf = pipes["Random Forest"].named_steps["model"]
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 4))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Top features — Random Forest")
plt.tight_layout()
plt.show()

## 6. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap

rf_pipe = pipes["Random Forest"]
X_train_scaled = rf_pipe.named_steps["scaler"].transform(X_train)
model = rf_pipe.named_steps["model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_scaled)

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values, X_train_scaled, feature_names=X.columns, max_display=8, show=False)
plt.title("SHAP Summary — Ventas")
plt.tight_layout()
plt.show()

## 7. Conclusiones

- Las ventas dependen de variables numéricas base.
- R² suele ser alto en este ejemplo.
- Se puede extender a datasets reales de retail con promociones, clima, etc.

**Rango esperado (ejemplo):** R² > 0.90